# AI-Assisted Data Analysis Using Agentic AI

I chose Claude as my AI platform to analyze a public dataset from Kaggle that provides information on coffee quality around the world and generate insights, visualizations, and a short analytical report.

## 1. Dataset Description

### 1.1 Source

The dataset was sourced from [Kaggle](https://www.kaggle.com/) (`df_arabica_clean.csv`) and contains arabica coffee quality reviews collected by the Coffee Quality Institute (CQI).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# I initially faced truncated output in Jupyter. Claude advised me to add these lines at the top of the notebook to remove all display limits.
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Load dataset
df = pd.read_csv('../data/df_arabica_clean.csv')

# Ensure visuals folder exists
os.makedirs('../visuals', exist_ok=True)

### 1.2 Size

The dataset contains 207 rows and 41 columns, each row representing a unique coffee sample.

In [ ]:
df.shape

### 1.3 Main Features/Variables

The dataset covers five main categories of variables:

- **Origin & Identity:** Country of Origin, Farm Name, Region, Producer, Variety, Processing Method
- **Logistics:** Harvest Year, Altitude, Number of Bags, Bag Weight
- **Taste Scores (scale ~6.5-10):** Aroma, Flavor, Aftertaste, Acidity, Body, Balance, Overall, Total Cup Points
- **Defects:** Category One Defects, Category Two Defects, Quakers
- **Certification:** Certification Body, Certification Address, Certification Contact

In [ ]:
# Column names, types, and null counts
df.info()

## 2. AI-Assisted Analysis

I used AI to guide me through each step of the analysis from exploring the dataset structure to generating visualizations. Claude suggested the approaches, and I implemented the code ad verified the results against the actual data.

### 2.1 Explore the Dataset

I started by exploring the dataset to understand its structure and the distribution of categorical variables such as country of origin and processing method. I also checked whether any owner submitted more than one variety of coffee, finding that Coffee Quality Union leads with 14 distinct varieties and Taiwan Coffee Laboratory with 12.

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Countries represented in the dataset
df['Country of Origin'].value_counts()

In [ ]:
# Processing methods used
df['Processing Method'].value_counts()

In [ ]:
# Owners with more than one coffee variety
owner_variety = df.groupby('Owner')['Variety'].nunique()
owner_variety[owner_variety > 1].sort_values(ascending=False)

### 2.2 Detect Missing Values

I found that most columns are complete, but a few have notable gaps like `ICO Number`, which is missing for the majority of entries.

> **What is an ICO Number?** The ICO (International Coffee Organization) Number is a tracking code assigned to coffee lots for international trade and export purposes. Its absence in most rows suggests many samples were submitted outside the formal ICO export pipeline, or the field was simply not recorded.

In [ ]:
# Count and percentage of missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

Based on this, `ICO Number` should be dropped from any further analysis due to 63.77% missing data; retaining it would either require imputation (which would be misleading for a tracking code) or silently exclude two-thirds of the dataset. The remaining columns with missing values (`Variety`, `Processing Method`, `Mill`, etc.) all have under 3% missing and can be kept as-is, since they are not used in the core scoring analysis.

### 2.3 Generate Statistics

This is where I started making real assumptions about the data. Looking at how taste scores were distributed, which countries consistently produced higher-rated coffee, and how frequently defects appeared across samples. Rather than just reading the data, I began forming insights.

#### 2.3.1 Taste Score Statistics

The taste scores are fairly tightly distributed, with `Total Cup Points` ranging from 78 to 89.33 and a mean of around 83.7.

In [ ]:
# Descriptive statistics for taste score columns
taste_cols = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Balance', 'Overall', 'Total Cup Points']
df[taste_cols].describe()

#### 2.3.2 Average Score by Country and Continent

I calculated the average Total Cup Points per country and per continent to see which origins produce the highest-rated coffee.

In [ ]:
# Average Total Cup Points by Country of Origin
df.groupby('Country of Origin')['Total Cup Points'].mean().sort_values(ascending=False)

In [ ]:
# Map countries to continents
continent_map = {
    'Colombia': 'South America', 'Brazil': 'South America', 'Peru': 'South America', 'Ecuador': 'South America',
    'Costa Rica': 'North America', 'Guatemala': 'North America', 'Honduras': 'North America',
    'Mexico': 'North America', 'Nicaragua': 'North America', 'Panama': 'North America',
    'El Salvador': 'North America', 'Haiti': 'North America',
    'United States (Hawaii)': 'North America', 'United States (Puerto Rico)': 'North America',
    'Ethiopia': 'Africa', 'Kenya': 'Africa', 'Uganda': 'Africa',
    'Tanzania, United Republic Of': 'Africa', 'Madagascar': 'Africa',
    'Burundi': 'Africa', 'Rwanda': 'Africa',
    'Taiwan': 'Asia', 'Laos': 'Asia', 'Vietnam': 'Asia', 'Myanmar': 'Asia',
    'Indonesia': 'Asia', 'India': 'Asia', 'Thailand': 'Asia',
    'Papua New Guinea': 'Oceania', 'Philippines': 'Asia',
}
df['Continent'] = df['Country of Origin'].map(continent_map)

# Average score by continent
df.groupby('Continent')['Total Cup Points'].agg(['mean', 'count']).round(2).sort_values('mean', ascending=False)

#### 2.3.3 Defect Frequency

I explored how often defects appear across samples.

> **What are Category One Defects?** These are primary (full) defects severe issues like black beans, sour/fermented beans, fungus-damaged beans, or foreign matter. Each instance counts as one full defect and has a strong negative impact on quality.

> **What are Category Two Defects?** These are secondary (partial) defects less severe issues like immature beans, broken or chipped beans, floaters, parchment, or slight insect damage. Multiple affected beans may be needed to count as a single defect.

These are not dataset-specific terms they are a globally standardized grading system defined by the **Specialty Coffee Association (SCA)**. To qualify as specialty grade, a coffee must have **zero Category One defects** and **fewer than 5 Category Two defects** in a 350g sample.

Category One defects are rare in this dataset 93% of samples have none at all. Category Two defects are more common and vary widely, with some samples having as many as 16.

In [ ]:
# Category One Defects frequency
df['Category One Defects'].value_counts().sort_index()

In [ ]:
# Category Two Defects frequency
df['Category Two Defects'].value_counts().sort_index()

### 2.4 Find Correlations and Trends

I asked Claude to help me identify which variables are most strongly related to the overall quality score.

#### 2.4.1 Taste Score Correlations

The results showed that `Flavor`, `Aftertaste`, and `Balance` have the strongest positive relationships with `Total Cup Points`.

In [ ]:
# Correlation of each taste score with Total Cup Points
df[taste_cols].corr()['Total Cup Points'].drop('Total Cup Points').sort_values(ascending=False)

#### 2.4.2 Quakers and Their Effect on Score

> **What are Quakers?** Quakers are underdeveloped coffee beans that fail to roast properly, staying pale or yellowish while other beans darken. They produce flat, peanut-like, or bland flavors and are considered a defect in specialty coffee grading.

I investigated whether the number of quakers affects the Total Cup Points. The correlation came out at -0.32, a weak negative relationship. Samples with 0 quakers averaged 83.88 points, while samples with 12 quakers dropped to 78.08 showing that higher quaker counts tend to lower the score, though the effect is moderate.

In [ ]:
# Average Total Cup Points by number of quakers
df.groupby('Quakers')['Total Cup Points'].agg(['mean', 'count']).round(2)

In [ ]:
# Correlation between quakers and Total Cup Points
df['Quakers'].corr(df['Total Cup Points']).round(4)

### 2.5 Create Visualizations

I worked with Claude to generate several visualizations to better understand the distribution of quality scores, the top-performing countries, and the relationships between variables. All charts are saved to the `visuals/` folder.

In [ ]:
# Distribution of Total Cup Points
plt.figure(figsize=(8, 4))
sns.histplot(df['Total Cup Points'], bins=20, kde=True, color='brown')
plt.title('Distribution of Total Cup Points')
plt.xlabel('Total Cup Points')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../visuals/distribution_total_cup_points.png', dpi=150)
plt.show()

In [ ]:
# Average Total Cup Points by Country (top 10)
top_countries = df.groupby('Country of Origin')['Total Cup Points'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='YlOrBr')
plt.title('Top 10 Countries by Average Total Cup Points')
plt.xlabel('Average Total Cup Points')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig('../visuals/top10_countries_avg_score.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap of taste scores
plt.figure(figsize=(10, 7))
sns.heatmap(df[taste_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap of Taste Scores')
plt.tight_layout()
plt.savefig('../visuals/correlation_heatmap_taste_scores.png', dpi=150)
plt.show()

In [ ]:
# Flavor vs Total Cup Points
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='Flavor', y='Total Cup Points', alpha=0.6, color='saddlebrown')
plt.title('Flavor vs Total Cup Points')
plt.xlabel('Flavor Score')
plt.ylabel('Total Cup Points')
plt.tight_layout()
plt.savefig('../visuals/flavor_vs_total_cup_points.png', dpi=150)
plt.show()

In [ ]:
# Average Total Cup Points by Continent
continent_avg = df.groupby('Continent')['Total Cup Points'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(8, 4))
sns.barplot(data=continent_avg, x='Total Cup Points', y='Continent', palette='YlOrBr')
plt.title('Average Total Cup Points by Continent')
plt.xlabel('Average Total Cup Points')
plt.ylabel('Continent')
plt.tight_layout()
plt.savefig('../visuals/continent_avg_score.png', dpi=150)
plt.show()

In [ ]:
# Average Total Cup Points by number of quakers
quaker_avg = df.groupby('Quakers')['Total Cup Points'].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=quaker_avg, x='Quakers', y='Total Cup Points', palette='YlOrBr')
plt.title('Average Total Cup Points by Quaker Count')
plt.xlabel('Number of Quakers')
plt.ylabel('Average Total Cup Points')
plt.tight_layout()
plt.savefig('../visuals/quakers_vs_total_cup_points.png', dpi=150)
plt.show()

## 3. Insights

After completing the analysis with Claude's assistance, I identified the following key insights from the dataset.

### Insight 1: Overall is the Strongest Predictor of Total Cup Points

Among all taste attributes, `Overall` has the highest correlation with `Total Cup Points` (0.947), followed closely by `Flavor` at 0.939. Graders and producers should prioritize overall impression and flavor development as they have the most direct impact on the final quality score.

### Insight 2: Scores Are Tightly Clustered, High-Scorers Stand Out

The majority of samples score between 82 and 86, with a mean of 83.7. The distribution is slightly left-skewed, meaning a few exceptional samples pull the average up. Any sample scoring above 87 is a clear outlier and represents top-tier specialty coffee.

### Insight 3: Absent ICO Numbers Suggest a Boutique-Skewed Dataset

The `ICO Number` column is missing for 63.77% of entries (132 out of 207). The absence of ICO tracking in 64% of samples suggests most coffees in this dataset entered the grading pipeline outside formal international trade channels, which may indicate a bias toward competition or boutique submissions rather than commercial export lots. This makes the dataset valuable for understanding high-quality specialty coffee, but limits its representativeness of the global commercial coffee market.

### Insight 4: Category One Defects Are Rare, Category Two Are Not

93% of samples have zero Category One defects, suggesting that major defects are screened out before grading. However, Category Two defects appear in the majority of samples and range from 0 to 16. This makes Category Two defects a more meaningful quality differentiator within this dataset.

### Insight 5: Quakers Negatively Affect the Score

There is a weak negative correlation (-0.32) between quaker count and Total Cup Points. Samples with no quakers averaged 83.88 points while samples with 12 quakers dropped to 78.08. Although the relationship is not strong, it confirms that underdeveloped beans consistently drag down the overall quality score.

### Insight 6: A Few Large Organizations Dominate the Dataset

Coffee Quality Union submitted 14 different varieties and Taiwan Coffee Laboratory submitted 12, while most other owners submitted only one or two. This suggests these are large-scale certification organizations rather than individual producers. It also made me realize that much of the competition in the dataset could be happening within or between just two or three major companies, creating a kind of internal hierarchy where the same organizations are both submitting and dominating the top-scoring entries.

### Insight 7: Africa Leads by Continent Ethiopia is the Best, El Salvador the Worst

I mapped each country to its continent and compared average Total Cup Points. Africa comes out on top with an average of 84.63, driven by strong performances from Ethiopia (84.96) and Tanzania (84.74). Asia follows at 84.00, largely led by Taiwan (84.35). North America and South America trail behind at 83.33 and 83.09 respectively.

At the country level, Ethiopia ranks first with an average of 84.96, making it the highest-rated origin in the dataset. El Salvador sits at the bottom with 81.53. It is worth noting that some countries like Madagascar and Myanmar only have one sample each, so their averages are not statistically reliable.

## 4. Reflection

### 4.1 Which AI Tool I Used

I used Claude by Anthropic as my AI assistant throughout this project. Before I even started writing code, Claude reviewed the dataset with me to confirm it contained enough valuable and varied data to work with. From there, Claude assisted me at each stage by helping me think through it.

### 4.2 Advantages

- **Domain knowledge:** Claude could explain concepts like quakers, SCA grading standards, and ICO numbers without me needing to look them up separately.
- **Dataset validation upfront:** Claude looked over the CSV before I began to confirm the data was worth analyzing and pointed out which columns would be most useful.
- **Code collaboration:** I wrote the code myself, and Claude helped me combine and extend it for example, building the country-to-continent mapping to extract continent-level insights from country data.
- **Grammatical and structural feedback:** Claude reviewed my markdown writing and helped me clean up wording and fix grammatical mistakes throughout the notebook.
- **Folder and project structure:** Claude helped me set up and maintain a clean project structure keeping the notebooks, data, visuals, and report organized in a logical way.
- **Iterative collaboration:** I could ask follow-up questions mid-session and Claude would adjust without losing context.

### 4.3 Limitations

- **File corruption risk:** On a few occasions, using command-line tools to edit the notebook file directly caused JSON corruption, requiring the file to be rebuilt from scratch.
- **No visual feedback:** Claude could not see the generated charts, so I had to open and review them myself to confirm they looked correct.
- **Required active involvement:** The collaboration only worked because I stayed engaged throughout Claude assisted, but every decision and verification was mine.
- **Cannot execute code:** Claude could not run cells I ran everything in Jupyter and verified the outputs independently.
- **Assumes clean data:** Claude's suggested code sometimes assumed column names or data types were consistent, which required checking against the actual dataset.
- **Configuration limitations:** Claude could not directly fix environment-level issues for example, when Jupyter was truncating output rows by default, Claude could only advise me on what settings to change; I had to apply the fix myself.

### 4.4 How I Verified AI-Generated Results

I verified Claude's outputs through several methods:

- **Running the code myself:** Every code cell was executed in Jupyter and I compared the output against what Claude described. If the numbers did not match the explanation, I flagged it.
- **Cross-checking statistics:** For key figures like the mean Total Cup Points (83.7), the top country (Ethiopia, 84.96), and the correlation values, I re-ran the relevant cells independently to confirm.
- **Checking definitions:** For domain-specific terms like SCA grading standards and quakers, I verified Claude's explanations against external sources to ensure accuracy before including them in the report.
- **Reviewing visualizations:** Each saved chart was opened and visually inspected to confirm it matched the data and that titles and labels were correct.